# Udemy Lesson Summarizer

Paste any Udemy lesson URL and this notebook will:
1. Extract the course and lecture identifiers from the URL
2. Fetch the lesson transcript via the Udemy API
3. Send it to OpenAI and return a structured summary

In [ ]:
# This cell verifies that all packages are available in the current environment.

import importlib

required = ["openai", "dotenv", "requests"]

for package in required:
    if importlib.util.find_spec(package) is None:
        print(f"✖︎ Missing: {package}")
    else:
        print(f"✔︎ Found: {package}")

In [ ]:
import os
import re
import requests

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

UDEMY_TOKEN = os.getenv("UDEMY_TOKEN")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

assert UDEMY_TOKEN, "✖︎ UDEMY_TOKEN not found in .env"
assert OPENAI_API_KEY, "✖︎ OPENAI_API_KEY not found in .env"

print("✔︎ Keys loaded successfully")

client = OpenAI(api_key=OPENAI_API_KEY)

## URL Parser

A Udemy lesson URL looks like this:
`https://www.udemy.com/course/python-bootcamp/learn/lecture/12345678`

We need to extract two pieces of information:
- `course_slug` → `python-bootcamp` (used to identify the course)
- `lecture_id` → `12345678` (used to identify the specific lesson)

In [ ]:
def parse_udemy_url(url: str) -> tuple[str, str]:
    """
    Extracts the course slug and lecture ID from a Udemy lesson URL.

    Args:
        url: Full Udemy lesson URL from the browser address bar.

    Returns:
        A tuple of (course_slug, lecture_id).

    Raises:
        ValueError: If the URL format is not recognized.
    """

    # This pattern matches the two dynamic segments in a Udemy lesson URL.
    # ([^/]+) captures any characters that are not a slash → course slug
    # (\d+)   captures a sequence of digits → lecture ID
    pattern = r"udemy\.com/course/([^/]+)/learn/lecture/(\d+)"
    match = re.search(pattern, url)

    if not match:
        raise ValueError(
            "✖︎ Invalid URL format. "
            "Expected: https://www.udemy.com/course/<name>/learn/lecture/<id>"
        )

    course_slug = match.group(1)  # first capture group
    lecture_id = match.group(2)   # second capture group

    print(f"✔︎ Course: {course_slug}")
    print(f"✔︎ Lecture: {lecture_id}")

    return course_slug, lecture_id

## Transcript Fetcher

Udemy generates automatic captions for most video lessons.  
We fetch them via the Udemy API using your Bearer token for authentication.

The transcript is delivered in **WebVTT format** - a standard subtitle format
that includes timestamps. We strip the timestamps and keep only the spoken text.

In [ ]:
def get_lecture_transcript(course_slug: str, lecture_id: str) -> tuple[str, str]:
    """
    Fetches the lesson transcript from the Udemy API.

    Args:
        course_slug: The course identifier from the URL.
        lecture_id:  The lecture identifier from the URL.

    Returns:
        A tuple of (lecture_title, clean_transcript_text).

    Raises:
        PermissionError: On authentication or access failures.
        ValueError:      If no captions are available for this lesson.
    """

    # Every request to the Udemy API must include this header.
    # The Bearer token proves that you are a logged-in user with course access.
    headers = {
        "Authorization": f"Bearer {UDEMY_TOKEN}",
        "Content-Type": "application/json",
    }

    # The Udemy API requires a numeric ID for subscription-based endpoints.
    course_response = requests.get(
        f"https://www.udemy.com/api-2.0/courses/{course_slug}/",
        headers=headers
    )

    # Handle the most common failure modes explicitly with clear messages
    if course_response.status_code == 401:
        raise PermissionError("✖︎ Token expired or invalid. Grab a fresh one from DevTools.")
    if course_response.status_code == 403:
        raise PermissionError("✖︎ Access denied. Make sure you are enrolled in this course.")

    # For any other non-2xx response, raise a generic HTTP error
    course_response.raise_for_status()
    course_id = course_response.json()["id"]

    # The `fields` query parameters tell the API to return only what we need,
    # keeping the response small and fast.
    lecture_response = requests.get(
        f"https://www.udemy.com/api-2.0/users/me/subscribed-courses/"
        f"{course_id}/lectures/{lecture_id}/"
        f"?fields[lecture]=asset,title"
        f"&fields[asset]=captions",
        headers=headers
    )

    lecture_response.raise_for_status()
    lecture_data = lecture_response.json()
    title = lecture_data.get("title", "Untitled Lecture")

    print(f"Lecture title: {title}")

    captions = lecture_data.get("asset", {}).get("captions", [])

    if not captions:
        raise ValueError(
            "✖︎ No captions found for this lesson. "
            "Try a different lecture - not all have transcripts."
        )

    # Prefer English captions; fall back to the first available language
    caption = next(
        (c for c in captions if c.get("locale_id", "").startswith("en")),
        captions[0]
    )

    print(f"Caption language: {caption.get('locale_id', 'unknown')}")

    # Step 3: Download the VTT file (plain HTTP, no auth needed - it's a CDN URL)
    vtt_response = requests.get(caption["url"])
    vtt_response.raise_for_status()

    # Step 4: Clean the WebVTT format.
    # A VTT file looks like this:
    #   WEBVTT
    #   1
    #   00:00:01.000 --> 00:00:04.000
    #   Hello and welcome to this course.
    #
    # We only want the spoken text lines - everything else gets discarded.
    clean_lines = []

    for line in vtt_response.text.splitlines():
        line = line.strip()

        if not line:
            continue                        # skip blank lines
        if line.startswith("WEBVTT"):
            continue                        # skip file header
        if "-->" in line:
            continue                        # skip timestamp lines
        if line.isdigit():
            continue  

        clean_lines.append(line)

    transcript = " ".join(clean_lines)
    
    print(f"✔︎ Transcript ready ({len(transcript):,} characters)")

    return title, transcript

## AI Summarizer

We send the cleaned transcript to OpenAI's API and ask for a structured summary.

**Model:** `gpt-4o-mini` - accurate, fast, and cheap. Perfect for this task.  
**Temperature:** `0.3` - lower values make the output more factual and consistent.

In [ ]:
def summarize_lecture(title: str, transcript: str) -> str:
    """
    Sends the transcript to OpenAI and returns a structured summary.

    Args:
        title:      The lesson title (included in the prompt for context).
        transcript: The cleaned transcript text.

    Returns:
        A formatted summary string from the model.
    """
    
    # GPT models have a context limit. We cap the transcript at 100,000 characters
    # to stay safely within bounds and control cost.
    MAX_CHARS = 100_000

    if len(transcript) > MAX_CHARS:
        transcript = transcript[:MAX_CHARS]
        print(f"Transcript trimmed to {MAX_CHARS:,} characters")

    # The prompt instructs the model on exactly what format we want.
    # Being explicit about structure produces consistent, usable output.
    prompt = f"""You are a learning assistant. Below is the transcript of an online lesson titled "{title}".

                Write a concise, practical summary in English with the following structure:

                1. **Lesson Goal** - What this lesson is about (2–3 sentences)
                2. **Key Concepts** - Bullet list of the most important terms or ideas
                3. **Main Takeaways** - 3-5 things worth remembering
                4. **Shorten content** - summary of lesson content
                5. **Practical Application** - How this knowledge can be applied

                Transcript:
                ---
                {transcript}
                ---
            """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,   # lower = more deterministic and factual
        max_tokens=10000,   # enough for a solid summary without overspending
    )

    # The response object contains a list of choices.
    # We always take the first one ([0]) as we only requested one completion.
    return response.choices[0].message.content

## Run It

Change `LESSON_URL` to any lesson you are enrolled in, then run this cell.

In [ ]:
LESSON_URL = "https://www.udemy.com/course/llm-engineering-master-ai-and-large-language-models/learn/lecture/52932255#overview"

# --- Pipeline execution ---
# Each function handles one responsibility: parse → fetch → summarize
# If any step fails, it raises a clear error and stops here.

print("Parsing URL...")
course_slug, lecture_id = parse_udemy_url(LESSON_URL)

print("\nFetching transcript...")
title, transcript = get_lecture_transcript(course_slug, lecture_id)

print("\nGenerating summary...")
summary = summarize_lecture(title, transcript)

# Display the final result
print("\n" + "=" * 60)
print(f"SUMMARY  {title}")
print("=" * 60)
print(summary)